In [3]:
pip install deap

In [5]:
import random
import numpy as np
from deap import base, creator, tools, algorithms
from sklearn.neural_network import MLPClassifier
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [7]:
data = load_iris()
X_train, X_test, y_train, y_test = train_test_split(
    data.data, data.target, test_size=0.3, random_state=42
)

In [9]:
# -----------------------------
# Fitness Function (Neural Network)
# -----------------------------
def evaluate(individual):
    neurons = individual[0]
    layers = individual[1]

    # Create hidden layer structure
    hidden_layer = tuple([neurons] * layers)

    try:
        model = MLPClassifier(hidden_layer_sizes=hidden_layer, max_iter=500)
        model.fit(X_train, y_train)
        predictions = model.predict(X_test)
        accuracy = accuracy_score(y_test, predictions)
    except:
        accuracy = 0  # in case model fails

    return (accuracy,)

In [11]:
# -----------------------------
# GA Setup
# -----------------------------
creator.create("FitnessMax", base.Fitness, weights=(1.0,))
creator.create("Individual", list, fitness=creator.FitnessMax)

toolbox = base.Toolbox()


In [13]:
# Parameters to optimize
toolbox.register("attr_neurons", random.randint, 5, 100)
toolbox.register("attr_layers", random.randint, 1, 3)

toolbox.register("individual", tools.initCycle, creator.Individual,
                 (toolbox.attr_neurons, toolbox.attr_layers), n=1)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)


In [15]:
# Operators
toolbox.register("evaluate", evaluate)
toolbox.register("mate", tools.cxTwoPoint)
toolbox.register("mutate", tools.mutUniformInt, low=5, up=100, indpb=0.2)
toolbox.register("select", tools.selTournament, tournsize=3)


In [17]:
# -----------------------------
# Run GA
# -----------------------------
population = toolbox.population(n=10)
generations = 5

for gen in range(generations):
    offspring = algorithms.varAnd(population, toolbox, cxpb=0.5, mutpb=0.2)
    fitnesses = list(map(toolbox.evaluate, offspring))

    for ind, fit in zip(offspring, fitnesses):
        ind.fitness.values = fit

    population = toolbox.select(offspring, k=len(population))

C:\Users\vaibhav\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
C:\Users\vaibhav\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


In [19]:
# -----------------------------
# Best Result
# -----------------------------
best_ind = tools.selBest(population, k=1)[0]
print("Best Parameters (Neurons, Layers):", best_ind)
print("Best Accuracy:", evaluate(best_ind)[0])

Best Parameters (Neurons, Layers): [73, 3]
Best Accuracy: 1.0
